In [ ]:
#phase 2 disocvery with pca and mefisto
favouritenumber = (67)
for i in range (1, 68):
  tothepower = (favouritenumber * i)
  print (tothepower)
  i = i + 1


print("star")
emptyspaces = (" ")
parts = ("67")
for i in range(3):
  for j in range(3):
    print (f"{emptyspaces * i * -j},{parts * i}, {emptyspaces * i}")

67
134
201
268
335
402
469
536
603
670
737
804
871
938
1005
1072
1139
1206
1273
1340
1407
1474
1541
1608
1675
1742
1809
1876
1943
2010
2077
2144
2211
2278
2345
2412
2479
2546
2613
2680
2747
2814
2881
2948
3015
3082
3149
3216
3283
3350
3417
3484
3551
3618
3685
3752
3819
3886
3953
4020
4087
4154
4221
4288
4355
4422
4489
star
,, 
,, 
,, 
,67,  
,67,  
,67,  
,6767,   
,6767,   
,6767,   


In [2]:
#data inegst establish path
# discovery with pca and mefisto
!pip install -q mofapy2 2>/dev/null.

from google.colab import drive
try:
    drive.mount('/content/drive')
except ValueError:
    drive.flush_and_unmount()
    drive.mount('/content/drive', force_remount=True)
import os
import json
import numpy as np
import pandas as pd
from collections import Counter
from datetime import datetime,timezone
fromthepast = 67
from sklearn.decomposition import PCA

#filling the paths source, database

sourcedata = '/content/drive/MyDrive/Glioma data'
database = '/content/drive/MyDrive/Tom_HLPA'
v1sourcedata = f'{database}'
specificationpath = f'{v1sourcedata}/IngestionStage/Specification.json'
mappedgenes = f'{v1sourcedata}/IngestionStage/Genemapping.json'
standadised = f'{database}/Standardised '
fromthelaststage = f'{database}/IngestionStage/manifest.json'
#gene and ml spec sheets
with open(specificationpath) as f:
  specification = json.load(f)
with open(mappedgenes) as f:
  mappedgenes = json.load(f)
#varaiables for next cell
topgenes = 5000
logoffset = 1
extremenegatives = -100
RandomSeed = specification['discovery']['random_state']
Ncomponents = specification['discovery']['n_components']
print (RandomSeed)
print (f'source = {sourcedata}')
print (f'specification = {specification}')
print (f'{standadised}')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.0 MB/s eta 0:00:00
Mounted at /content/drive
42
source = /content/drive/MyDrive/Glioma data
specification = {'version': '1.0', 'frozen_date': '2026-09-10', 'source_data_directory': '/content/drive/MyDrive/Glioma data', 'workspace': '/content/drive/MyDrive/HLP_Atlas_Workspace/v1.0', 'datasets': {'GLASS_GBM': {'expression': 'core_mrna.parquet', 'clinical': 'core_clin2.parquet', 'survival': 'core_ttr.parquet', 'longitudinal': True, 'modalities': ['mRNA'], 'model': 'MEFISTO', 'preprocessing': {'log_transform': 'NOT_APPLIED', 'log_transform_justification': 'Source matrix is already on a standardised scale (median per-gene std = 0.993). Applying log2(TPM+1) would be invalid.', 'duplicate_gene_handling': 'collapse by mean before selection', 'n_duplicate_rows_collapsed': 5, 'gene_filtering': 'drop all-NaN, constant, extreme-negative; keep top 5000 by variance', 'dropped_genes': {'all_nan': 487, 'constant_value': 24, 'extreme_negative': 

In [ ]:
manifestpath = f'{fromthelaststage}'
tcgaentries = []
with open(manifestpath) as f:
    manifest = json.load(f)

print(f"Successful datasets: {manifest['n_successful']}")
print(f"Failed datasets: {manifest['n_failed']}")

for entry in manifest['datasets_processed']:
    expr_path = f"{standadised}/{entry['output_expression']}"

    assert os.path.exists(expr_path), f"Missing: {entry['dataset']}"

    expr = pd.read_parquet(expr_path)

    assert np.isfinite(expr.values).all(), f"Non-finite values: {entry['dataset']}"
    assert expr.index.is_unique, f"Duplicate samples: {entry['dataset']}"
    assert expr.columns.is_unique, f"Duplicate genes: {entry['dataset']}"

    print(f"{entry['dataset']}: {expr.shape}")
    print(f"{entry['output_expression']}")


Successful datasets: 13
Failed datasets: 2
TCGA_BLCA: (428, 5000)
TCGA_BLCA_expression.parquet
TCGA_BRCA: (1226, 5000)
TCGA_BRCA_expression.parquet
TCGA_COAD: (514, 5000)
TCGA_COAD_expression.parquet
TCGA_KIRC: (610, 5000)
TCGA_KIRC_expression.parquet
TCGA_LIHC: (424, 5000)
TCGA_LIHC_expression.parquet
TCGA_LUAD: (589, 5000)
TCGA_LUAD_expression.parquet
TCGA_OV: (429, 5000)
TCGA_OV_expression.parquet
TCGA_PAAD: (183, 5000)
TCGA_PAAD_expression.parquet
TCGA_PRAD: (554, 5000)
TCGA_PRAD_expression.parquet
TCGA_SKCM: (473, 5000)
TCGA_SKCM_expression.parquet
TCGA_STAD: (448, 5000)
TCGA_STAD_expression.parquet
TCGA_UCEC: (585, 5000)
TCGA_UCEC_expression.parquet
GLASS_GBM: (247, 5000)
GLASS_GBM_expression.parquet


In [ ]:
# Load successful TCGA entries from the ingestion manifest
tcgaentries = [
    entry
    for entry in manifest["datasets_processed"]
    if entry["dataset"].startswith("TCGA_")
]

print(f"TCGA cohorts found: {len(tcgaentries)}")
for entry in tcgaentries:
    print(entry["dataset"], "->", entry["output_expression"])

TCGA cohorts found: 12
TCGA_BLCA -> TCGA_BLCA_expression.parquet
TCGA_BRCA -> TCGA_BRCA_expression.parquet
TCGA_COAD -> TCGA_COAD_expression.parquet
TCGA_KIRC -> TCGA_KIRC_expression.parquet
TCGA_LIHC -> TCGA_LIHC_expression.parquet
TCGA_LUAD -> TCGA_LUAD_expression.parquet
TCGA_OV -> TCGA_OV_expression.parquet
TCGA_PAAD -> TCGA_PAAD_expression.parquet
TCGA_PRAD -> TCGA_PRAD_expression.parquet
TCGA_SKCM -> TCGA_SKCM_expression.parquet
TCGA_STAD -> TCGA_STAD_expression.parquet
TCGA_UCEC -> TCGA_UCEC_expression.parquet


In [ ]:
'''keep this one'''
# PCA for tcga
import gc

tcgamanifets = []

for entry in tcgaentries:
    datasetid = entry["dataset"]
    expressionpath = f"{database}/Standardised /{entry['output_expression']}"

    try:
        expression = pd.read_parquet(expressionpath)

        nsmaples = expression.shape[0]
        ngenes = expression.shape[1]

        print(f"Input: {datasetid}, Samples: {nsmaples}, Genes: {ngenes}")

        pca = PCA(n_components=Ncomponents, random_state=RandomSEed)

        scores = pca.fit_transform(expression.values)
        loadings = pca.components_.T
        explained = pca.explained_variance_ratio_

        print(f"Components: {scores.shape[1]}")
        print(f"Cumulative variance: {explained.sum() * 100:.2f}%")

        factornames = [f"Factor_{i+1}" for i in range(Ncomponents)]

        scoresDF = pd.DataFrame(
            scores,
            index=expression.index,
            columns=factornames
        )
        scoresDF.index.name = "sampleID"

        loadingsDF = pd.DataFrame(
            loadings,
            index=expression.columns,
            columns=factornames
        )
        loadingsDF.index.name = "gene"

        outscores = f"{database}/Discovery/{datasetid}_scores.parquet"
        outloadings = f"{database}/Discovery/{datasetid}_loadings.parquet"

        scoresDF.to_parquet(outscores)
        loadingsDF.to_parquet(outloadings)

        # Readback verification
        v_scores = pd.read_parquet(outscores)
        v_loadings = pd.read_parquet(outloadings)

        assert v_scores.shape == scoresDF.shape
        assert v_loadings.shape == loadingsDF.shape
        assert list(v_scores.columns) == list(scoresDF.columns)
        assert list(v_loadings.columns) == list(loadingsDF.columns)
        assert list(v_scores.index) == list(scoresDF.index)
        assert list(v_loadings.index) == list(loadingsDF.index)

        meta = {
            "dataset": datasetid,
            "model": "PCA",
            "model_class": "SINGLE_VIEW_FACTOR_MODEL",
            "n_components": Ncomponents,
            "random_state": RandomSEed,
            "n_samples": int(nsmaples),
            "n_genes": int(ngenes),
            "variance_explained": [float(v) for v in explained],
            "cumulative_variance": float(explained.sum()),
            "longitudinal": False,
            "output_scores": os.path.basename(outscores),
            "output_loadings": os.path.basename(outloadings),
            "fitted_at": datetime.now(timezone.utc).isoformat(),
        }

        with open(f"{database}/Discovery/{datasetid}_model_metadata.json", "w") as f:
            json.dump(meta, f, indent=2)

        tcgamanifets.append(meta)
        print("Saved and verified successfully.")

        del expression, scores, loadings, scoresDF, loadingsDF, v_scores, v_loadings, pca
        gc.collect()

    except Exception as e:
        import traceback
        print(f"{datasetid} failed: {type(e).__name__}: {e}")
        traceback.print_exc()

print(f"Successful: {len(tcgamanifets)}")
print(f"Failed: {len(tcgaentries) - len(tcgamanifets)}")


Input: TCGA_BLCA, Samples: 428, Genes: 5000
Components: 30
Cumulative variance: 61.03%
Saved and verified successfully.
Input: TCGA_BRCA, Samples: 1226, Genes: 5000
Components: 30
Cumulative variance: 58.48%
Saved and verified successfully.
Input: TCGA_COAD, Samples: 514, Genes: 5000
Components: 30
Cumulative variance: 64.23%
Saved and verified successfully.
Input: TCGA_KIRC, Samples: 610, Genes: 5000
Components: 30
Cumulative variance: 68.17%
Saved and verified successfully.
Input: TCGA_LIHC, Samples: 424, Genes: 5000
Components: 30
Cumulative variance: 63.79%
Saved and verified successfully.
Input: TCGA_LUAD, Samples: 589, Genes: 5000
Components: 30
Cumulative variance: 61.81%
Saved and verified successfully.
Input: TCGA_OV, Samples: 429, Genes: 5000
Components: 30
Cumulative variance: 53.97%
Saved and verified successfully.
Input: TCGA_PAAD, Samples: 183, Genes: 5000
Components: 30
Cumulative variance: 67.98%
Saved and verified successfully.
Input: TCGA_PRAD, Samples: 554, Genes: 50

In [ ]:

def build_temporal(meta, timeline):
    timeline["START_DATE_num"] = pd.to_numeric(
        timeline["START_DATE"], errors="coerce"
    )
    timeline_indexed = timeline.set_index("SAMPLE_ID")

    temporal = pd.DataFrame({
        "sample_id": meta.index,
        "patient_id": meta["patient_id"].values,
        "timepoint": meta["timepoint"].values,
        "start_date": timeline_indexed.reindex(meta.index)["START_DATE_num"].values,
    })

    temporal = temporal.set_index("sample_id")
    temporal.index.name = "sample_id"
    temporal["elapsed_days"] = np.nan

    for pid, group in temporal.groupby("patient_id"):
        baseline = group["start_date"].min()
        temporal.loc[group.index, "elapsed_days"] = (
            group["start_date"] - baseline
        )

    return temporal


def check_glass(expr, temporal):
    checks = {
        "expression 247x5000": expr.shape == (247, 5000),
        "all finite": bool(np.isfinite(expr.values).all()),
        "unique genes": expr.columns.is_unique,
        "unique samples": expr.index.is_unique,
        "no zero var": int((expr.var(axis=0) == 0).sum()) == 0,
        "temporal aligned": temporal.index.equals(expr.index),
        "temporal complete": bool(temporal["elapsed_days"].notna().all()),
        "temporal has variation": bool(temporal["elapsed_days"].std() > 0),
        "patients 118": temporal["patient_id"].nunique() == 118,
        "all multi-sample": bool(
            (temporal.groupby("patient_id").size() > 1).all()
        ),
    }

    return checks


In [5]:

# Build GLASS temporal covariate

expr = pd.read_parquet(
    f"{database}/Standardised /GLASS_GBM_expression.parquet"
)
meta = pd.read_parquet(
    f"{database}/Standardised /GLASS_GBM_metadata.parquet"
)

timeline_indexed = timeline.set_index("SAMPLE_ID")

temporal = pd.DataFrame({
    "sample_id": meta.index,
    "patient_id": meta["patient_id"].values,
    "timepoint": meta["timepoint"].values,
    "start_date": timeline_indexed.reindex(meta.index)["START_DATE"].values,
})

temporal = temporal.set_index("sample_id")
temporal.index.name = "sample_id"

temporal["elapsed_days"] = np.nan

for pid, group in temporal.groupby("patient_id"):
    baseline = group["start_date"].min()
    temporal.loc[group.index, "elapsed_days"] = (
        group["start_date"] - baseline
    )

print("Expression samples:", len(expr))
print("Temporal samples:", len(temporal))
print("Missing start dates:", temporal["start_date"].isna().sum())
print("Missing elapsed days:", temporal["elapsed_days"].isna().sum())
print("Elapsed days range:",
      temporal["elapsed_days"].min(),
      "to",
      temporal["elapsed_days"].max())
print("Unique patients:", temporal["patient_id"].nunique())

temporal_out = temporal[
    ["patient_id", "timepoint", "start_date", "elapsed_days"]
].copy()

temporal_out.to_parquet(
    f"{database}/Standardised /GLASS_GBM_temporal.parquet"
)

print("Saved GLASS_GBM_temporal.parquet")


Expression samples: 247
Temporal samples: 247
Missing start dates: 0
Missing elapsed days: 0
Elapsed days range: 0.0 to 3772.0
Unique patients: 118
Saved GLASS_GBM_temporal.parquet


In [ ]:
'''Glass mefisto aufarm'''
from mofapy2.run.entry_point import entry_point

expression = pd.read_parquet(f"{database}/Standardised /Glass_GBM_expression.parquet")
temporal = pd.read_parquet(f"{database}/Standardised /GLASS_GBM_temporal.parquet").reindex(expression.index)
temporalcovariates = temporal['elapsed_days'].values.astype(np.float32)

data = [[expression.values.astype(float32)]]
samplenames = [list(expression.index)]
featurenames = [list(expression.columns)]
groupname = ['group1']

'''fit mofa the fun and tuff part''
ent = entry_point()
ent.set_data_options(scale_views=False)
ent.set_data_matrix(
    data,
    views_names=['mRNA'],
    groups_names=groups_names,
    samples_names=samples_names,
    features_names=features_names,
)
ent.set_model_options(
    factors=N_COMPONENTS,
    spikeslab_weights=True,
    ard_weights=True,
)
ent.set_train_options(
    iter=500,
    convergence_mode='fast',
    seed=RANDOM_SEED,
    verbose=False,
    quiet=False,
)
ent.set_covariates(
    sample_cov=[time_covariate.reshape(-1, 1)],
    covariates_names=['elapsed_days'],
)

ent.set_smooth_options(
    scale_cov=True,
    start_opt=20,
    n_grid=10,
    opt_freq=10,
    model_groups=True,
    warping=False,
    sparseGP=True,
    frac_inducing=0.75,
)

print("\nRunning MEFISTO...\n")
ent.build()
ent.run()
print("\nTraining complete.")


# Extract Z (factor scores)
Z = np.asarray(ent.model.getExpectations()['Z']['E'])

if Z.shape[0] == N_COMPONENTS:
    scores = Z.T
elif Z.shape[1] == N_COMPONENTS:
    scores = Z
else:
    raise ValueError(f"Unexpected Z shape: {Z.shape}")

assert scores.shape == (len(expr.index), N_COMPONENTS)


# Extract W (factor loadings)
W_raw = ent.model.getExpectations()['W']

if isinstance(W_raw, dict):
    entry = W_raw['mRNA']
    W_mrna = np.asarray(entry['E'] if isinstance(entry, dict) else entry)

elif isinstance(W_raw, list):
    first = W_raw[0]
    if isinstance(first, dict):
        W_mrna = np.asarray(first['E'])
    elif isinstance(first, list):
        inner = first[0]
        W_mrna = np.asarray(inner['E'] if isinstance(inner, dict) else inner)
    else:
        W_mrna = np.asarray(first)
else:
    raise TypeError(f"Unexpected W structure: {type(W_raw)}")

if W_mrna.shape[0] == expr.shape[1]:
    loadings = W_mrna
elif W_mrna.shape[1] == expr.shape[1]:
    loadings = W_mrna.T
else:
    raise ValueError(f"W shape {W_mrna.shape} doesn't match genes={expr.shape[1]}")

assert loadings.shape == (expr.shape[1], N_COMPONENTS)


# Save scores and loadings
factor_names = [f"Factor_{i+1}" for i in range(N_COMPONENTS)]

scores_df = pd.DataFrame(scores, index=expr.index, columns=factor_names)
scores_df.index.name = 'sample_id'

loadings_df = pd.DataFrame(loadings, index=expr.columns, columns=factor_names)
loadings_df.index.name = 'gene'

out_scores = f'{DISC_DIR}/GLASS_GBM_scores.parquet'
out_loadings = f'{DISC_DIR}/GLASS_GBM_loadings.parquet'

scores_df.to_parquet(out_scores)
loadings_df.to_parquet(out_loadings)

# Verify saved files
v_scores = pd.read_parquet(out_scores)
v_loadings = pd.read_parquet(out_loadings)

assert v_scores.shape == scores_df.shape
assert v_loadings.shape == loadings_df.shape
assert v_scores.index.equals(scores_df.index)
assert v_loadings.index.equals(loadings_df.index)
assert v_scores.columns.equals(scores_df.columns)
assert v_loadings.columns.equals(loadings_df.columns)

print("Scores:", scores_df.shape)
print("Loadings:", loadings_df.shape)


# Save metadata
meta_out = {
    "dataset": "GLASS_GBM",
    "requested_model": "MEFISTO",
    "actual_model": "MEFISTO",
    "mofapy2_version": "0.7.5",
    "n_components": N_COMPONENTS,
    "random_state": RANDOM_SEED,
    "n_samples": int(expr.shape[0]),
    "n_genes": int(expr.shape[1]),
    "n_patients": int(temporal['patient_id'].nunique()),
    "longitudinal": True,
    "temporal_covariate": {
        "name": "elapsed_days",
        "definition": "Days from patient's earliest observed tumour sample",
        "source": "GLASS core_timeline.parquet / START_DATE",
        "range_days": {
            "min": int(time_covariate.min()),
            "max": int(time_covariate.max()),
            "median": int(np.median(time_covariate)),
        },
    },
    "smooth_options": {
        "scale_cov": True,
        "sparseGP": True,
        "frac_inducing": 0.75,
        "model_groups": True,
        "warping": False,
    },
    "training": {
        "iter": 500,
        "convergence_mode": "fast",
    },
    "timepoint_distribution": {
        str(k): int(v)
        for k, v in temporal['timepoint'].value_counts().items()
    },
    "output_scores": os.path.basename(out_scores),
    "output_loadings": os.path.basename(out_loadings),
    "fitted_at": datetime.now(timezone.utc).isoformat(),
}

meta_path = f'{DISC_DIR}/GLASS_GBM_model_metadata.json'

with open(meta_path, 'w') as f:
    json.dump(meta_out, f, indent=2)

print("Metadata saved:", meta_path)
print("GLASS MEFISTO complete.")


        #########################################################
        ###           __  __  ____  ______                    ### 
        ###          |  \/  |/ __ \|  ____/\    _             ### 
        ###          | \  / | |  | | |__ /  \ _| |_           ### 
        ###          | |\/| | |  | |  __/ /\ \_   _|          ###
        ###          | |  | | |__| | | / ____ \|_|            ###
        ###          |_|  |_|\____/|_|/_/    \_\              ###
        ###                                                   ### 
        ######################################################### 
         


Successfully loaded view='mRNA' group='group1' with N=247 samples and D=5000 features...


Model options:
- Automatic Relevance Determination prior on the factors: False
- Automatic Relevance Determination prior on the weights: True
- Spike-and-slab prior on the factors: False
- Spike-and-slab prior on the weights: True
Likelihoods:
- View 0 (mRNA): gaussian


Loaded 1 covariate(s) for